In [1]:
import json
import traceback
from pathlib import Path
import numpy as np

import aiida
from aiida import orm

from tqdm import tqdm

from mc3d_data_utils import verify_optimization_convergence, get_base_details_entry

AIIDA_PROFILE_NAME = "mc3d-pbesol-v2"

aiida.load_profile(AIIDA_PROFILE_NAME)

LOG_DST = Path().resolve() / "log"

# delete old log:
if LOG_DST.exists():
    LOG_DST.unlink()

def log(msg):
    print(msg)
    with open(LOG_DST, 'a') as f:
        f.write(str(msg) + '\n')

In [2]:
# list aiida groups
! verdi -p {AIIDA_PROFILE_NAME} group list --all --all-users --count

  PK  Label                   Type string    User               Node count
----  ----------------------  -------------  ---------------  ------------
   7  20250908-144201         core.import    aiida@localhost       1126488
   2  magnetic_information    core           aiida@localhost         33142
   4  structures/final        core           aiida@localhost         33142
   5  stuctures/final/unique  core           aiida@localhost         32013
   3  workchain/cod/relax     core           aiida@localhost         11780
   6  workchain/icsd/last-pw  core           aiida@localhost         15980
   1  workchain/mpds/last-pw  core           aiida@localhost          5382


In [ ]:
# Generate node graph image for a random PWCalculation to understand data layout
! verdi -p {AIIDA_PROFILE_NAME} node graph generate --process-out --process-in -f svg "31ba8f73-b92b-4ace-9d81-5a736361e7f0" -O graph.svg

# We see that
# * The final structure can have PwBaseWorkChain and PwRelaxWorkChain parents
# * the final parameters are associated with the PwBaseWorkChain parent
# * magnetic parameters are produced from the retrieved FolderData node via a calcfunction

In [3]:
import aiida.plugins

PwBaseWorkChain = aiida.plugins.WorkflowFactory("quantumespresso.pw.base")

qb = (
    orm.QueryBuilder()
    .append(orm.Group, filters={"label": "structures/final"}, tag="group")
    .append(orm.StructureData, with_group="group", project=["*"], tag="structure")
    .append(PwBaseWorkChain, with_outgoing="structure", project=["*"], tag="wc")
    .append(
        orm.ArrayData,
        with_incoming="wc",
        edge_filters={"label": "output_trajectory"},
        project=["*"],
    )
    .append(
        orm.Dict,
        with_incoming="wc",
        edge_filters={"label": "output_parameters"},
        project=["*"],
        tag="parameters",
    )
    .append(orm.CalcJobNode, with_outgoing="parameters", project=["*"], tag="final_opt_calc")
    .append(orm.FolderData, with_incoming="wc", edge_filters={"label": "retrieved"}, tag="fd")
    .append(orm.CalcFunctionNode, with_incoming="fd", tag="cj")
    .append(orm.Dict, with_incoming="cj", tag="magnetic_params", project=["*"])
)

count = qb.distinct().count()
print(f"Number of structures: {count}")

Number of structures: 33134


In [4]:
data = {}

count_successful = 0
count_skipped = 0

for structure, workchain, opt_traj, out_params, final_opt_calc, magnetic_params in tqdm(qb.iterall()):
    try:

        if verify_optimization_convergence(opt_traj) is None:
            log(f"SKIPPING: Thresholds not met for workchain {workchain}")
            count_skipped += 1
            continue

        extras = structure.base.extras.all
        mc3d_id = extras.get('mc3d_id')

        if not mc3d_id:
            log(f"SKIPPING: MC3D ID not assigned to {extras['source']}")
            count_skipped += 1
            continue

        details_entry = get_base_details_entry(structure, [out_params, magnetic_params])
        details_entry["provenance_links"] = [
            {"label": "Final relax calculation", "uuid": final_opt_calc.uuid},
            {"label": "Final structure", "uuid": structure.uuid},
        ]
        data[mc3d_id] = details_entry

        count_successful += 1

    except Exception as exception:
        log(f"Found exception for {structure}: {exception}")
        traceback.print_exc()
        count_skipped += 1
        break


log(f"Successful entries {count_successful}")
log(f"Skipped entries {count_skipped}")

OUTPUT_PATH = Path().resolve().parent

with (OUTPUT_PATH / "base.json").open("w") as handle:
    json.dump(data, handle, sort_keys=True, indent=2)

0it [00:00, ?it/s]/home/eimre_k/mcxd/discover-mc3d-data/common-library/mc3d_data_utils/__init__.py:39: AiidaDeprecationWarning: `StructureData.extras` is deprecated, use `StructureData.base.extras.all` instead. (this will be removed in v3)
  extras = structure_node.extras
99it [00:05, 67.58it/s]/home/eimre_k/micromamba/envs/aiida/lib/python3.10/site-packages/pymatgen/core/composition.py:1365: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])
782it [00:14, 83.68it/s]/home/eimre_k/micromamba/envs/aiida/lib/python3.10/site-packages/pymatgen/core/composition.py:1365: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  syms: list[str] = sorted(sym_amt, key=lambda x: [get_el_sp(x).X, x])


Successful entries 33134
Skipped entries 0
